# Day 3: Forms, tools and a score

**Every record is well-formed. What would convince you the records are right?**

The short answer the day works towards is: not their shape.

The full reading for the day, with its sources: [the course web page](https://catalyte-ai-irs-bpa-to6-weeks.vercel.app/week-01/#day-3)

### Today: from text to something a program can use.

**Day 2**

- A language model produces its answer one piece at a time
- That is why it can be fluent and wrong
- The system prompt, revised when it failed
- The model and its reasoning level, chosen with a reason

**How today runs**

- A reply checked against a schema
- A reply that names a function for your code to run
- A set of cases with a score the prompt is held to
- That score judges day 2's revisions and model choice

Section 2.2 wrote the extractor's system prompt and revised it when it failed; section 2.3 chose the model and its reasoning level and wrote down why.

## 3.1 Ask for structured output

*If the model's reply is guaranteed to match your schema, is it correct?*

Each Section plan shows the whole day; walk it once here.

### A schema guarantees shape, not content; it can cost accuracy.

**What it buys**

- A reply your code can rely on
- No required key left out
- No invalid enum value (enum: a field's fixed list of values)
- It limits which tokens may come at each step

**What it does not**

- OpenAI: “Structured Outputs can still contain mistakes.”
- Input unrelated to the schema can push it to hallucinate
- No source shows it makes a model invent less

**What it can cost**

- A 2024 study: reasoning declined under format restrictions
- Part of it: GPT 3.5 Turbo put the “answer” key before “reason”
- So measure the format on the evaluation set

The sources behind this chapter support something narrower. One vendor, OpenAI, says a schema means you need not worry about the model omitting a required key or hallucinating an invalid enum value; enum is the JSON Schema keyword for a field with a fixed list of values. One study found format restrictions helped on some classification tasks. Neither shows that a schema makes a model less likely to fill in a value the input does not contain. The paper behind Outlines, an open-source library for constraining model output, speaks of guaranteeing the structure of the generated text: structure, not content. OpenAI's guide says outright that Structured Outputs can still contain mistakes, and that the model will always try to adhere to the schema, which can result in hallucinations if the input is completely unrelated to it. What structured output does buy is a reply your code can rely on: Anthropic's page describes valid, parseable output for downstream processing.

A 2024 study of five models found a significant decline in reasoning abilities under format restrictions, and traced part of it to the order of keys: 100% of GPT 3.5 Turbo's JSON-mode responses placed the answer key before the reason key.

### Every field is required, and only the amount may be null.

*Null is what lets a note with no amount come back without an invented one.*

| Field | Type | May be null |
| --- | --- | --- |
| tax_year | integer | No |
| form | string | No |
| issue | string | No |
| amount_in_question | integer or null | Yes: a note may name none |
| next_action | string | No |

This is the case-notes extractor's schema, written in JSON Schema, the standard way to write what a valid reply looks like. Only amount_in_question may be null, because the lab's notes always state the other four fields. A field that can be absent in your own data needs null too.

The extractor's contract is three things: required, a type that allows null, and no extra keys. With required, an object is valid only if every name in the array is a property in it. For a value that may be absent, keep the field required and allow null; OpenAI's guide says it is possible to emulate an optional parameter by using a union type with null.

*Drawn: one field of the schema, with and without null; both keep it required.*

<img src="figures/null-lets-no-amount-pass.png" width="864" alt="Two columns for the field amount_in_question. Integer only: required; for a note that names an amount, the amount; for a note that names no amount, one valid move, put a number there, so the invented amount is the only answer that passes. Integer or null, marked in orange: required; for a note that names an amount, the amount; for a note that names no amount, null passes, so the reply can say there is none.">

*Allowing null in amount_in_question is what lets a note with no amount come back without an invented one.*

The figure draws the extractor's schema for one field, amount_in_question, with and without null. The null matters more than it looks. If amount_in_question were required and could only be an integer, a note with no amount would leave the model one valid move: put a number there.

*Source: the chapter, section 3.1, the extractor's schema; OpenAI, Structured Outputs guide*

### Validate every reply in your own code anyway.

*The guarantee has stated exceptions.*

- May fail the schema: a refusal, a cut-off, capitalization
- Tax year 2203 passes the schema, not your own check

<img src="figures/the-schema-then-your-own-checks.png" width="864" alt="A reply comes in from the left to a blue box, The schema; an arrow labelled passes leads on to an orange box, Your own check. From each box an arrow labelled fails drops to one band along the bottom: Counted as a failure, never repaired by guessing. Beside the schema&#x27;s arrow, in blue, what it stops: a refusal; cut off at the token limit; an enum value with different capitalization. Beside your own check&#x27;s arrow, in orange, what it stops: a tax year of 2203 is a valid integer.">

Anthropic's page says structured outputs guarantee schema compliance in most cases, and names three scenarios where the output may not match your schema: a refusal, a reply cut off at the token limit, and an enum value returned with different capitalization. And a reply can pass a schema and still be unusable; the JSON Schema specification says structural validation alone may be insufficient for an application to use certain values correctly. A tax year of 2203 is a valid integer, so only the extractor's own check, the orange gate, stops it. So the extractor validates every reply against the schema and against simple checks of its own, and a reply that fails is counted as a failure, never repaired by guessing.

*Source: Anthropic, Structured outputs; the JSON Schema specification; the chapter, section 3.1*

**Check:** In the 2024 study, “answer” came first. What did that cost?

- **A.** Nothing: the reply still matched the format
- **B.** It answered directly instead of reasoning first
- **C.** It left out required keys

<details><summary>Answer</summary>

**B.** With “answer” first, it answered before it reasoned: format can cost accuracy, so measure it.

</details>

**Discuss:** Guaranteed shape, guaranteed right?

- If the reply is guaranteed to match your schema, is it correct?
- Why may amount_in_question be null, and what if it could not be?
- If the schema is guaranteed, why validate the reply in your own code?

---

### Exercise: Check replies against a schema and your own check

After this exercise, you can check each reply against the schema and your own check, and count a bad reply as a failure.

**Scenario:** Five of eight replies failed `check_reply` in `extractor.py`. So a teammate changed the schema and `check_reply`, and now two fail.

**Where you work:** the folder `~/practice/week-01/day-03/validate-replies/lab`, in VS Code. In the Explorer, it is `week-01 › day-03 › validate-replies › lab`. Each path in this exercise starts there. Do not use AI tools.

**Done when:** cell 3 prints `Done:`.

**1. See the start.** Run cell 1. The first time, it builds `lab`: the lab after its run 2, with the teammate's commit on top. It prints `validate-replies is back as it was supplied.` In `lab`, you have:

- `schema/output.schema.json` and `extractor.py`, as the teammate changed them
- `replies/`, eight replies, each named for its note in `notes/`
- `count_failures.py`, which gives each reply to `check_reply`, and counts the failures

The course made up every note. The rules are in section 3.1 above, "Ask for structured output".

Cell 1 opens the schema and `extractor.py`. In the Explorer, they are under `week-01 › day-03 › validate-replies › lab`.

To start again, run `!python3 ~/practice/reset.py validate-replies` in a new cell. It removes your changes.

In [ ]:
# 1. Start the exercise. Open the two files.
!test -d ~/practice/week-01/day-03/validate-replies/lab/.git || python3 ~/practice/reset.py validate-replies
%cd -q ~/practice/week-01/day-03/validate-replies/lab
!code schema/output.schema.json extractor.py

**2. Make the three changes by hand, in order.** First run cell 2. You should see `Failures: 2 of 8 replies.`

| Change | Write this | Why |
|---|---|---|
| 1. `next_action` | In `schema/output.schema.json`, make line 10 `"next_action": {"type": "string"}` | Note 0138 ends "Wait for the call.", but reply 0138 gives null. Every note states its next action, so null there is wrong. Section 3.1, below the table of fields: "Only amount_in_question may be null, because the lab's notes always state the other four fields." |
| 2. An amount as text | In `extractor.py`, remove lines 29 to 34, from `# The model sometimes` | These lines turn "40,000 of 95,000" in reply 0105 into 4000095000, a wrong amount. A guess can be wrong, so the reply must fail instead. Section 3.1, below the second figure: "a reply that fails is counted as a failure, never repaired by guessing" |
| 3. Your own check | In `extractor.py`, put `import datetime` above `import json`. Put the lines below just above `return fields`, after the schema check | Reply 0130 gives tax year 2203, but note 0130 names 2023. The schema passes any integer, so only your own check can fail a year that has not begun. After the schema check, `tax_year` is an integer, so `>` can compare it with this year. Section 3.1, in the list above the second figure: "Tax year 2203 passes the schema, not your own check" |

```python
    if fields["tax_year"] > datetime.date.today().year:
        raise ExtractionFailed("the tax year is after this year")
```

Save each file. Then run cell 2 again. You should see `Failures: 6 of 8 replies.`

In [ ]:
# 2. Count the failures.
%cd -q ~/practice/week-01/day-03/validate-replies/lab
!python3 count_failures.py

**3. Check.** Run cell 3. You should see `Done:`. If you see `Not yet:`, fix what it names. Then run cells 2 and 3 again.

In [ ]:
# 3. Check.
%cd -q ~/practice/week-01/day-03/validate-replies/lab
!python3 ../check.py

**Afterwards, together**

- The comment that you removed names "18,400", one clear number. Should `check_reply` turn it into 18400, and why?
- When would `next_action` need null too?
- Which wrong tax year does your own check still pass, and what would stop it?

<details><summary>Answer</summary>

`~/practice/week-01/day-03/answers/validate-replies/` holds the fixed files, `decisions.md`, and `commands.sh`, which ends with `Done:`.

</details>

The same exercise for the terminal: [`validate-replies/README.md`](validate-replies/README.md)

---

## 3.2 Describe a function

*When a model calls a function, who runs the function?*

### Your code runs the function; the model only asks.

*OpenAI: “When the model calls a function, you must execute it and return the result.”*

<img src="figures/who-runs-the-function.png" width="864" alt="Three steps, left to right. You describe the functions: a name, a description, a schema for the arguments. The model, in grey, replies with a call: it only asks. Your code runs it, in orange, and decides what each call may do. A dashed line returns from your code to the model: your code returns the result.">

The round trip, with the model in grey and your code in orange: you describe the functions, each with a name, a description and a schema for its arguments; the model replies with a call; your code runs it and sends the result back. The call is a request that does nothing until your code acts on it.

*Source: OpenAI, Function calling guide; the chapter, section 3.2*

### That makes the function list a boundary.

*The model can ask only for what you defined.*

- Your code decides what each call may do
- A value the model must not choose stays in your code

<img src="figures/a-value-the-model-must-not-choose.png" width="864" alt="Two sides: The model, in grey, at the left; Your code at the right. Between them a blue dashed line, the function list, with one definition sitting on it in each row. Top row: submit_refund(order_id). A grey arrow, a call, runs into it from the model&#x27;s side carrying order_id, in ink: the model fills it. Bottom row: submit_refund(). A grey arrow, a call, runs into it from the model&#x27;s side carrying nothing; an orange arrow brings order_id into it from your code&#x27;s side: your code passes it.">

The same guide gives a related habit for arguments your code already knows: define submit_refund() with no parameters and pass the order_id in your code. The guide's reason is to offload work from the model. The same move keeps a value the model must not choose out of its hands, which is what the figure's two rows show: in the top row the model fills order_id, in the bottom row your code passes it.

*Source: OpenAI, Function calling guide; the chapter, section 3.2*

### A lookup is harmless; an action on a case belongs to a person.

*IRS policy: “Do not use GenAI to make binding determinations or actions on taxpayer rights without proper oversight.”*

| A function the model may call | What it does | Offer it? |
| --- | --- | --- |
| Look up what a form number means | Returns the form's title; records nothing | Yes: harmless |
| Record an action on a case | Writes an action into the case file | No: that call belongs to a person |

For the case-notes extractor, a function the model can call to look up what a form number means is harmless.

### Describe it for a colleague; then watch for invented arguments.

**Describe each function**

- Its purpose, and each parameter
- Could an intern use it from that alone?
- strict: true, so calls adhere to the schema
- OpenAI: “We recommend always enabling strict mode.”

**Invented arguments**

- A valid call can carry a guessed value
- Anthropic: Claude Sonnet “might also infer a reasonable value”
- The Berkeley Function Calling Leaderboard, a public benchmark, also tests irrelevance: tools available, none invoked
- Your set: notes where a call is right, and notes where none is

Your evaluation set should hold both: notes where a call is right, and notes where no call should be made.

**Check:** Which call does the Berkeley leaderboard count as correct?

- **A.** Any call whose JSON is valid
- **B.** The name matches exactly, and every argument is an allowed value
- **C.** Any call made whenever tools are available

<details><summary>Answer</summary>

**B.** A valid call can still carry a guessed value. Test notes where a call is right, and notes where none is.

</details>

**Discuss:** Who runs the function?

- When a model calls a function, who runs the function?
- Why does that make the list of functions a boundary?
- Which functions should the extractor never offer the model, and why?

---

### Exercise: Define one function, and test both cases

After this exercise, you can describe a function for the model, and test it where a call is right and where none is.

**Scenario:** A teammate wrote the functions that the model can ask for when a caseworker asks about case notes. Fix the list before anyone uses it.

**Where you work:** the folder `~/practice/week-01/day-03/define-a-function/lab`, in the day's notebook or in a terminal. In the Explorer, it is week-01 › day-03 › define-a-function › lab. Each path starts there. `ask.py` calls the model. Use no other AI tool.

**Done when:** cell 3 prints `Done:`.

**1. Open the list.** Run cell 1. The first time, it builds the lab after the data-rules check, with a teammate's commit. It prints `define-a-function is back as it was supplied.` Then it opens `functions.json` and shows the name of each function. Or open it from the Explorer. You have:

- `functions.json`, the function list, which you change.
- `tools.py`, the code of each function.
- `ask.py`, which sends two questions and saves the replies in `replies/`.

The rules are in section 3.2 above, Describe a function.

To start again, run `!python3 ~/practice/reset.py define-a-function` in a new cell. The reset removes your changes and replies.

In [ ]:
# 1. Open the function list.
!test -d ~/practice/week-01/day-03/define-a-function/lab/.git || python3 ~/practice/reset.py define-a-function
%cd -q ~/practice/week-01/day-03/define-a-function/lab
!code functions.json
!grep '"name"' functions.json

**2. Fix the list by hand. Then send the two questions.** Make these three changes in `functions.json`. Copy each `\"`: JSON needs a `\` before a quote inside a text.

| Part | Write this | Why |
|---|---|---|
| 1. `add_follow_up` | Remove its definition, from its `{` to its `},`. | In `tools.py`, it writes the next action into the case file. IRS policy keeps such actions under a person's oversight. Section 3.2, the table of functions, row "Record an action on a case": "No: that call belongs to a person". |
| 2. The `description` of `lookup_form` | In place of `TODO`: `Gives the title of a tax form, such as \"U.S. Individual Income Tax Return\" for 1040. Call it only when a question asks what a form is, and the note does not say it. It reads a table of titles. It writes nothing.` | The model cannot see `tools.py`. So this text says what the function does, when to call it, and that it writes nothing. Notes 0106 and 0147 name a form. Only the question about 0106 asks what a form is. Section 3.2, list "Describe each function": "Could an intern use it from that alone?" |
| 3. The `description` of `form_number` | In place of `TODO`: `The number of the form as the note writes it, without the word Form, such as \"5329\", \"1120-S\" or \"W-2\".` | `tools.py` finds the title of `5329`, but nothing for `Form 5329`. So the description says what to send. Section 3.2, the same list: "Its purpose, and each parameter". |

Save `functions.json`. Then run cell 2, which runs `python3 ask.py`. You should see a call to `lookup_form` with `{"form_number": "5329"}` for note 0106, and no call for note 0147. If it prints `Not sent:`, fix what it names. For a list that it sent before, it sends nothing and shows the saved replies.

In [ ]:
# 2. Send the two questions with your function list.
%cd -q ~/practice/week-01/day-03/define-a-function/lab
!python3 ask.py

**3. Check.** Run cell 3. It tests your list and the two replies that cell 2 saved for it. You should see `Done:`. If you see `Not yet:`, fix what it names. Then run cell 2 again. Then run cell 3.

In [ ]:
# 3. Check.
%cd -q ~/practice/week-01/day-03/define-a-function/lab
!python3 ../check.py

**Afterwards, together**

- When could `add_follow_up` go back on the list, and why?
- A teammate wants to add `read_note(note_id)`, which reads any note. Who should choose `note_id`, and why?
- Which third question would you add to test the list, and why?

<details><summary>Answer</summary>

`~/practice/week-01/day-03/answers/define-a-function/` holds the finished list, the kept replies, `decisions.md` and `commands.sh`.

</details>

The same exercise for the terminal: [`define-a-function/README.md`](define-a-function/README.md)

---

## 3.3 Start the evaluation set

*Five notes right. How often is it right, and would you know next month?*

### Reading by eye fails in two ways; a labelled set fixes both.

*The answer that feels sufficient: I read the outputs and they looked fine.*

**Why reading by eye fails**

- Five is a handful, not a rate
- No memory: after next month's prompt edit, nothing tells you the five still pass
- Or that a sixth kind of note now fails

**The evaluation set**

- A fixed set of labelled cases, scored the same way every time
- From its first day, the likeliest to fail: no amount, two years, an ambiguous issue
- Easy notes that all pass measure nothing
- Small this week; it grows every week

Make the set hold failures, not just cases. Eugene Yan writes on evaluating language-model products: for pass or fail evaluations, most of the time, what matters is the fails, as these are the trust-busting defects.

### Start by looking, not by measuring.

*See outputs before you fully define criteria.*

- Each note beside its JSON and the JSON it should be
- A note on each wrong field; pass or fail, and why

<img src="figures/one-note-on-the-review-sheet.png" width="864" alt="A review sheet for Note 0212, a made-up lab note. Five fields in rows, each in blue, with two columns: produced and should be. tax_year, form, issue and next_action show the same grey value on both sides, joined by an equals sign. amount_in_question, on an orange band: produced 0, not equal to, should be null; beside it, in an orange box, the short note: invented: the note names none. At the foot, the label: pass, in grey, and fail, chosen, in white on black; why: the model put 0 in amount_in_question.">

Hamel Husain, writing from consulting work with more than thirty companies, calls error analysis the single most valuable activity in AI development. The order matters: you can't fully define your evaluation criteria until you've seen a wide range of outputs, but you need criteria to evaluate those outputs in the first place. His write-up reports client results without a published dataset, so take the method from it, not a benchmark. For the extractor, patterns appear quickly: amounts invented for notes that name none; a date in the follow-up line read as the tax year; an issue copied word for word rather than summarised.

Husain prefers a pass or fail label to a rating scale, whose middle points evaluators struggle to tell apart. A binary label does not lose the detail, because the detail has moved to the written critique that accompanies the judgment. Eugene Yan gives a starting format: a spreadsheet with columns for input, output, additional metadata that helps evaluate the output, and a new column for the label.

The chapter calls that a failure. Its cause, which the exercise's write-up asks for, is why it happened: the schema required the field and allowed only an integer, so the model had to put a number there.

*Source: Hamel Husain on error analysis; the chapter, section 3.3*

### Score each field, set the bar, and run it on every change.

**Score the fields, not the note**

- Precision: how many extracted values were correct
- Recall: how many expected values were found
- Per field, the score names the instruction to change

**Set the bar; run it on every change**

- Fields a person reads tolerate more misses
- Anthropic: an instruction passed weeks of evaluations; with other prompt changes, one broader evaluation then showed a 3% drop
- Its fix: evaluations per model, on every prompt change
- Open-source agent software: the prompt “tested in around 1% of cases”

Scoring each field separately is this course's choice for the lab, because it tells you which instruction in the prompt to change.

Yan: different use cases come with different levels of risk, so evaluation standards should be calibrated accordingly. A change that passes a narrow set can still be a regression. In its own incident report, Anthropic added a system prompt instruction to its coding tool to make answers shorter, after multiple weeks of internal testing and no regressions in the evaluations it ran. The report says the instruction did its damage in combination with other prompt changes, and does not say what the 3% is a percentage of. Its stated fix: run a broad suite of per-model evals, evaluations run for each model, for every system prompt change.

In open-source agent software, prompts are rarely tested: a study of unit tests sampled 377 test functions from 39 agent frameworks and 382 from 439 applications built on them, and found the prompt tested in around 1% of cases.

**Check:** A note names no amount; the reply invents one. Which drops?

- **A.** Recall: an expected value was not found
- **B.** Precision: an extracted value was not correct
- **C.** Neither: the reply still matches the schema

<details><summary>Answer</summary>

**B.** An invented amount on a note that names none lowers precision; a missed follow-up action lowers recall.

</details>

*The two measures separate the two ways an extractor can be wrong.*

<img src="figures/precision-and-recall.png" width="864" alt="Two columns. Precision, marked in orange: what proportion of extracted attributes were correct; in the lab&#x27;s scorer, how many extracted values were correct; lowered by an invented amount on a note that names none. Recall: what proportion of ground truth attributes were extracted; in the lab&#x27;s scorer, how many expected values were found; lowered by a missed follow-up action.">

*An invented amount lowers precision, the share of extracted values that were correct.*

The figure's questions are quoted from Eugene Yan; the lab's wording and the examples are this course's.

*Source: Eugene Yan, Task-Specific LLM Evals that Do & Don't Work, 2024; the chapter, section 3.3*

**Discuss:** How often is it right?

- What would you need before you could say how often it is right?
- How would you know next month that a change had made it worse?
- What is the difference between a failure and its cause?

---

### Exercise: Score the prompt on a labelled set, and revise it once

After this exercise, you can score a prompt per field on a labelled set, and revise it on one failure.

**Scenario:** A teammate changed the extractor's prompt. Five replies looked right. Nobody read its score on the set.

**Where you work:** the folder `~/practice/week-01/day-03/score-the-prompt/extractor`, in VS Code. In the Explorer, it is week-01 › day-03 › score-the-prompt › extractor. Each path starts there. `evaluation/score.py` calls the model. Use no other AI tool.

**Done when:** cell 4 prints `Done:`.

**1. See the first score.** Run cell 1. The first time, it builds `extractor`, the week's lab with the teammate's prompt. It prints `score-the-prompt is back as it was supplied.` Then it opens three files in the editor. They are also in the Explorer, under `extractor`: the teammate's prompt `prompts/system.txt`, `evaluation/runs/1/score.txt` and your write-up `evaluation/failure.md`. In `evaluation/runs/1/score.txt`, you should see `form` at 0.00 for precision and recall.

To start again, run `!python3 ~/practice/reset.py score-the-prompt` in a new cell. The reset removes your commits and runs.

In [ ]:
# 1. Start the exercise. Open the files.
!test -d ~/practice/week-01/day-03/score-the-prompt/extractor/.git || python3 ~/practice/reset.py score-the-prompt
%cd -q ~/practice/week-01/day-03/score-the-prompt/extractor
!code prompts/system.txt evaluation/runs/1/score.txt evaluation/failure.md

**2. Revise the prompt once.** Make the three changes of this table by hand. Save each file.

| Part | Write this | Why |
|---|---|---|
| 1. `Note:` and `Field:` in `evaluation/failure.md` | `Note: 0147` and `Field: form` | In `evaluation/runs/1/score.txt`, all 20 wrong values are in `form`. Each label gives the right value: the form number only, such as "1040". Each reply adds "Form". So any note shows the failure, such as 0147. Section 3.3, the list "Score the fields, not the note": "Per field, the score names the instruction to change" |
| 2. `Cause:` in `evaluation/failure.md` | `Cause: 11, 17` | Lines 11 and 17 of `prompts/system.txt` are the replies of the two examples. Both write `"form": "Form 1040"`. No line says how to write a form, so the model copies the examples. Day 2 showed the same with amounts. Day 2, section 2.2, the list "Examples teach the shape": "Every example with an amount teaches that notes have amounts" |
| 3. `prompts/system.txt` | In lines 11 and 17, write `"form": "1040"` in place of `"form": "Form 1040"`. | The model copies the examples. With "1040" in them, it writes the form number only, as the labels do. Day 2, section 2.2, under "Revise by editing the prompt, then re-score the whole set.": "A loop: run the set, score it, edit the prompt" |

Then run cell 2. For a new prompt, it sends the 20 notes to the model. You should see `form` above 0.00 in the line that ends with `(this prompt)`. If not, cell 4's check says what to do next. Before you save your change, cell 2 prints `Change it`.

In [ ]:
# 2. Score your prompt.
%cd -q ~/practice/week-01/day-03/score-the-prompt/extractor
!python3 evaluation/score.py

**3. Commit.** In cell 3, set `commit = True`. Then run cell 3. It commits your write-up, your prompt and its run. `-qm` keeps Git quiet and gives it the message, so no editor opens. When the commit works, it prints nothing. If no file changed, it prints `nothing to commit`. Before you set `commit`, it only tells you to set it.

In [ ]:
# 3. Commit, when commit is True.
%cd -q ~/practice/week-01/day-03/score-the-prompt/extractor
commit = False
if commit:
    !git add prompts/system.txt evaluation && git commit -qm "Revise the prompt once"
else:
    print("After step 2, set commit = True. Then run this cell again.")

**4. Check.** Run cell 4. You should see `Done:`. If you see `Not yet:`, fix what it names. Then run cells 2 to 4 again.

In [ ]:
# 4. Check.
%cd -q ~/practice/week-01/day-03/score-the-prompt/extractor
!python3 ../check.py

**Afterwards, together**

- Why can a labelled set find what the teammate's five replies did not?
- When would a new line about the form be a better change than the examples?
- Your second score is one run. Which differences come from your change, and which from the run?

<details><summary>Answer</summary>

`~/practice/week-01/day-03/answers/score-the-prompt/` holds the course's answer and its `decisions.md`.

</details>

The same exercise for the terminal: [`score-the-prompt/README.md`](score-the-prompt/README.md)

---

## Close

*Key points, and what day 4 builds*

### The reply is checked, the call is bounded, the prompt is scored.

- A schema guarantees shape, not content: validate every reply in your own code
- Allow null: without it, the invented amount is the only answer that passes
- Your code runs the function; the model only asks, so the list is a boundary
- The prompt is scored per field on a set that holds failures, on every change
- Write up a failure with its cause: the cause points at its own fix

### The prompt has a recorded score.

- Day 4: Git and the coding agent
- The repository, its remote copy and the diff
- The coding agents that write changes into them

Close the day. Day 4 turns to the repository, its remote copy and the diff, and the coding agents that write changes into them.